# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [46]:
import Bio.SeqIO
import Bio.Seq
seqreads = list(Bio.SeqIO.parse("barcodes_R1.fastq", format="fastq"))


NA = Bio.Seq.Seq("CACGATAGATAAATAATAGTGCACCAT")
HA = Bio.Seq.Seq("CCGGATTTGCATATAATGATGCACCAT")

reverse_NA = NA.reverse_complement()
reverse_HA = HA.reverse_complement()

print(reverse_NA)
print(reverse_HA)
print(len(reverse_HA))

ATGGTGCACTATTATTTATCTATCGTG
ATGGTGCATCATTATATGCAAATCCGG
27


In [47]:
#Taken from Lecture 09: 
# Test cases for determining HA, and NA correctly
test_cases_gene = [
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTG", "NA", "positive: Determining NA correctly"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTATGGTGCATCATTATATGCAAATCCGG", "HA", "positive: Determining HA correctly"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTATGGTGCACTATTATTTATCTATCGT", None, "No gene found"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTATGGTGCATCATTATATGCAAATCCG", None, "No gene found"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTF", None, "No gene found"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTATGGTGCATCATTATATGCAAATCCGF", None, "No gene found"),
]

In [48]:
#Taken from Lecture 09: 
#helper for running test cases for determining HA, and NA correctly
def run_tests_gene(parser_function, test_cases):
    n_failed = 0
    
    for seq, expected, label in test_cases:
        result = parser_function(seq)
        if result == expected:
            print(f"PASS  {label}")
        else:
            n_failed += 1
            print(f"FAIL  {label}: {seq!r} gave {result!r}, expected {expected!r}")
    
    return f"{n_failed} tests failed"


In [49]:
import re

#Taken from Lecture 09: 
#define read_gene funcion for determining HA and NA correctly
def read_gene(seqread, genelen=27, downstream="AGGCGGCCGC"):

    #Copied from the print reverse complement of NA and HA sequences above 
    NA ="CACGATAGATAAATAATAGTGCACCAT"
    HA ="CCGGATTTGCATATAATGATGCACCAT"

    

    # Flip the reverse-orientation read into forward orientation (gene + downstream)
    seqread_upper = str(seqread).upper()
    seqread_original = str(Bio.Seq.Seq(seqread_upper).reverse_complement())
    gene_pattern = re.compile(f"(?P<gene>[ACTGN]{{{genelen}}}){downstream}")
    match = gene_pattern.search(seqread_original)

    #to troubleshoot
    # print("input: ", seqread_upper)
    # print("flipped: ", seqread_original)
    # print("pattern: ", barcode_pattern.pattern)
    # print("match: ", match)
    # print("barcode:", match.group("barcode"))

#matching if the matched gene is equal to the NA and HA sequences. 
    if match is None:
        return None
    elif match.group("gene") == NA:
        return "NA"
    elif match.group("gene") == HA:
        return "HA"
    else:
        return None



#same from lecture 9

def read_barcode(seqread, bclen=16, upstream="AGGCGGCCGC"):
  
    # Flip the reverse-orientation read into forward orientation (upstream + barcode)
    seqread_upper = str(seqread).upper()
    seqread_original = str(Bio. Seq. Seq(seqread_upper).reverse_complement())
    barcode_pattern = re.compile(f"{upstream}(?P<barcode>[ACTGN]{{{bclen}}}$)")
    match = barcode_pattern.search(seqread_original)

    if match:
        return match.group("barcode")
    else:
        return None 




In [50]:
#question 1: how many reads for NA and HA are in the file?
#Taken from Lecture 09: 
NA_counts = 0
HA_counts = 0
unmatched_gene = 0
unmatched_barcode = 0

for read in seqreads:

    #checked which gene the reads were from, and kept a counter for each
    gene = read_gene(read.seq)
    if gene is None:
        unmatched +=1 
    
    elif gene == "NA":
        barcode = read_barcode(read.seq)
        if barcode:
            NA_counts += 1
        else:
            unmatched_barcode += 1        
    elif gene == "HA":
        barcode = read_barcode(read.seq)
        if barcode:
            HA_counts += 1
        else:
            unmatched_barcode += 1


print ("The answer to question 1: ")
print ("Number of reads that map to NA: ", NA_counts)
print ("Number of reads that map to NA: ", HA_counts)
#print ("Number of reads w/o gene: ", unmatched_gene)
#print ("Number of reads w/o barcode: ", unmatched_barcode)
#print ("Sum: ", NA_counts + HA_counts + unmatched_gene + unmatched_barcode)


The answer to question 1: 
Number of reads that map to NA:  3910
Number of reads that map to NA:  5246


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [51]:
#question 2: 
#same base code useed for question 1, only needed to add an extra vaiable.   
NA_counts = 0
HA_counts = 0
unmatched_barcode_NA = 0
unmatched_barcode_HA = 0

for read in seqreads:
    gene = read_gene(read.seq)
    if gene is None:
        pass
    # nested the read barcode for each "if gene" statement, so that unmatched barcoded could be counted, split by gene
    elif gene == "NA":
        barcode = read_barcode(read.seq)
        if barcode:
            NA_counts += 1
        else:
            unmatched_barcode_NA += 1        
    elif gene == "HA":
        barcode = read_barcode(read.seq)
        if barcode:
            HA_counts += 1
        else:
            unmatched_barcode_HA += 1

print ("The answer to question 2: ")
print ("Number of reads that map to HA but dont have a valid barcode: ", unmatched_barcode_HA)
print ("Number of reads that map to NA but dont have a valid barcode: ", unmatched_barcode_NA)

The answer to question 2: 
Number of reads that map to HA but dont have a valid barcode:  70
Number of reads that map to NA but dont have a valid barcode:  63


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [52]:
# Qurstion 3: Simiarly used the solution from step 4 of lecture 9. get function was indetifed using google as a way to parse through the dictionary to check if the barcode already existed. 
# for the max function, claude assisted in identifing the need for the key parameter to idenitify the barcode associated with the max value. 
NA_barcode_counts = {}
HA_barcode_counts = {}

#same as step 4 of the lecture, generating a dictionary for every uniqu barcode, adding +1 every time it's repeated, sperated by genes like described above. 
for read in seqreads:
    gene = read_gene(read.seq)
    if gene is None:
        pass  
    elif gene == "NA":
        barcode = read_barcode(read.seq)
        if barcode:
            NA_barcode_counts[barcode] = NA_barcode_counts.get(barcode, 0) + 1
        else:
            pass        
    elif gene == "HA":
        barcode = read_barcode(read.seq)
        if barcode:
            HA_barcode_counts[barcode] = HA_barcode_counts.get(barcode, 0) + 1
        else:
            pass

#claude assisted in identifiying the use of key parameter to find the barcode from using the max function.
max_NA_barcode = max(NA_barcode_counts, key=NA_barcode_counts.get)
max_HA_barcode= max(HA_barcode_counts, key=HA_barcode_counts.get)

print ("The answer to question 3: ")
print("The HA barcode with the most counts is: ", max_HA_barcode, "And the number of counts is:", max(HA_barcode_counts.values()))
print("The NA barcode with the most counts is: ", max_NA_barcode, "And the number of counts is:", max(NA_barcode_counts.values()))


The answer to question 3: 
The HA barcode with the most counts is:  CCCGACCCGACATTAA And the number of counts is: 155
The NA barcode with the most counts is:  ACCAGTTCTCCCCGGG And the number of counts is: 152
